# télos MDLM: Benchmarking Suite
This notebook contains the various benchmarking suites to profile model throughput, kernel latency, and deep memory breakdowns.

## Throughput Benchmark
Measures Tokens/sec and overall throughput statistics.

In [1]:
"""Multi-Size MDLM Throughput Benchmark Suite with Explicit Metal Memory Garbage Collection.

Uses Apple MLX for native Metal zero-leak Unified Memory execution.

Supports Model Sizes: 5M, 25M, 125M

Usage:
  python scripts/benchmark_throughput.py --model-size 25M --warmup 3 --steps 15
  python scripts/benchmark_throughput.py --model-size 125M --warmup 2 --steps 10
"""

import sys
import time
import argparse
from pathlib import Path
import numpy as np
import mlx.core as mx
import mlx.nn as nn
import mlx.optimizers as optim

sys.path.insert(0, "..")


MODEL_CONFIGS = {
    "5M": {"d_model": 256, "n_layers": 4, "n_heads": 4},
    "25M": {"d_model": 512, "n_layers": 8, "n_heads": 8},
    "125M": {"d_model": 768, "n_layers": 12, "n_heads": 12},
}


class MLXRMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-5):
        super().__init__()
        self.weight = mx.ones((d_model,))
        self.eps = eps

    def __call__(self, x):
        variance = mx.mean(mx.square(x), axis=-1, keepdims=True)
        return x * mx.rsqrt(variance + self.eps) * self.weight


class MLXSwiGLU(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        hidden = int(d_model * 8 / 3)
        hidden = ((hidden + 63) // 64) * 64
        self.w1 = nn.Linear(d_model, hidden, bias=False)
        self.w2 = nn.Linear(d_model, hidden, bias=False)
        self.w3 = nn.Linear(hidden, d_model, bias=False)

    def __call__(self, x):
        return self.w3(nn.silu(self.w1(x)) * self.w2(x))


class MLXBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        self.norm1 = MLXRMSNorm(d_model)
        self.norm2 = MLXRMSNorm(d_model)
        self.head_dim = d_model // n_heads
        self.n_heads = n_heads

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out = nn.Linear(d_model, d_model, bias=False)
        self.mlp = MLXSwiGLU(d_model)

    def __call__(self, x):
        B, T, D = x.shape
        h = self.norm1(x)

        q = self.q_proj(h).reshape(B, T, self.n_heads, self.head_dim).transpose(0, 2, 1, 3)
        k = self.k_proj(h).reshape(B, T, self.n_heads, self.head_dim).transpose(0, 2, 1, 3)
        v = self.v_proj(h).reshape(B, T, self.n_heads, self.head_dim).transpose(0, 2, 1, 3)

        scale = 1.0 / (self.head_dim ** 0.5)
        out = mx.fast.scaled_dot_product_attention(q, k, v, scale=scale)
        out = out.transpose(0, 2, 1, 3).reshape(B, T, D)

        x = x + self.out(out)
        x = x + self.mlp(self.norm2(x))
        return x


class MLXTelosTransformer(nn.Module):
    def __init__(self, vocab_size=8192, d_model=256, n_layers=4, n_heads=4):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model)
        self.layers = [MLXBlock(d_model, n_heads) for _ in range(n_layers)]
        self.norm = MLXRMSNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def __call__(self, x):
        x = self.emb(x)
        for layer in self.layers:
            x = layer(x)
        return self.head(self.norm(x))


def sample_beta_timesteps(batch_size: int, eps: float = 1e-5):
    u = mx.random.uniform(0.0, 1.0, (batch_size, 1))
    t = 0.5 - 0.5 * mx.cos(np.pi * u)
    return mx.clip(t, eps, 1.0)


def apply_masking_mlx(input_ids, mask_token_id=1):
    B, T = input_ids.shape
    t_values = sample_beta_timesteps(B)
    rand_matrix = mx.random.uniform(0.0, 1.0, (B, T))
    mask_positions = rand_matrix < t_values
    masked_input_ids = mx.where(mask_positions, mask_token_id, input_ids)
    return masked_input_ids, mask_positions, t_values


def loss_fn(model, masked_input_ids, targets, mask_positions, t_values):
    logits = model(masked_input_ids)
    B, T, V = logits.shape

    logits_flat = logits.reshape(-1, V)
    targets_flat = targets.reshape(-1)

    ce_per_token = nn.losses.cross_entropy(logits_flat, targets_flat, reduction="none").reshape(B, T)
    masked_ce = ce_per_token * mask_positions.astype(mx.float32)

    masked_count = mx.clip(mx.sum(mask_positions.astype(mx.float32), axis=1), 1.0, float(T))
    per_example_ce = mx.sum(masked_ce, axis=1) / masked_count
    unweighted_ce = mx.mean(per_example_ce)

    t_weights = 1.0 / mx.clip(mx.squeeze(t_values, -1), 1e-3, 1.0)
    reweighted_loss = mx.mean(per_example_ce * t_weights)
    return reweighted_loss, unweighted_ce


def main():
    parser = argparse.ArgumentParser(description="Multi-Size MDLM Throughput Benchmark Suite")
    parser.add_argument("--model-size", type=str, choices=["5M", "25M", "125M"], default="25M", help="Model size ('5M', '25M', '125M')")
    parser.add_argument("--warmup", type=int, default=3, help="Warmup steps (default: 3)")
    parser.add_argument("--steps", type=int, default=15, help="Timed benchmark steps (default: 15)")
    args = parser.parse_known_args()[0]

    model_size_key = args.model_size
    m_cfg = MODEL_CONFIGS[model_size_key]
    warmup_steps = args.warmup
    num_steps = args.steps

    print("\n" + "=" * 85)
    print(f" TELOS MDLM — {model_size_key} MODEL FAST BENCHMARK SUITE (Apple MLX Metal)")
    print(f" Warmup Steps: {warmup_steps} | Benchmark Steps: {num_steps} | Metal Cache GC: Active")
    print(f" Architecture: d_model={m_cfg['d_model']}, layers={m_cfg['n_layers']}, heads={m_cfg['n_heads']}, seq_len=512")
    print("=" * 85)

    dummy_model = MLXTelosTransformer(**m_cfg)
    import gc
    gc.collect()
    mx.clear_cache()
    param_count = sum(p.size for p in tree_flatten(dummy_model.parameters()))
    del dummy_model
    import gc
    gc.collect()
    mx.clear_cache()
    print(f" {model_size_key} Model Parameter Count: {param_count:,}")

    start_bench = time.perf_counter()

    seq_len = 512
    batch_sizes = [8, 16, 32] if model_size_key == "125M" else [16, 32, 64]
    latencies = {}

    # 1. Microbatch Size Sweep (BF16 Baseline)
    print("\n" + "=" * 85)
    print(f" 1. MICROBATCH SIZE SWEEP (BF16 Baseline, Warmup={warmup_steps}, Steps={num_steps})")
    print("=" * 85)
    print(f"{'BATCH SIZE':<12} | {'STEPS/SEC':<12} | {'TOKENS/SEC':<15} | {'LATENCY/STEP (ms)':<18}")
    print("-" * 85)

    for bs in batch_sizes:
        model = MLXTelosTransformer(**m_cfg)
        model.set_dtype(mx.bfloat16)
        mx.eval(model.parameters())

        optimizer = optim.AdamW(learning_rate=3e-4)
        loss_and_grad_fn = nn.value_and_grad(model, loss_fn)
        
        def compiled_step_uncompiled(tgts):
            masked_ids, mask_pos, t_vals = apply_masking_mlx(tgts)
            (loss, _), grads = loss_and_grad_fn(model, masked_ids, tgts, mask_pos, t_vals)
            optimizer.update(model, grads)
            return loss

        targets = mx.random.randint(0, 8192, (bs, seq_len))
        mx.eval(targets)

        # Warmup and initialize state
        for _ in range(warmup_steps):
            loss = compiled_step_uncompiled(targets)
            mx.eval(model.parameters(), optimizer.state, loss)

        state = [model.state, optimizer.state]
        compiled_step = mx.compile(compiled_step_uncompiled, inputs=state, outputs=state)
        mx.clear_cache()

        # Timed benchmark steps
        start = time.perf_counter()
        for _ in range(num_steps):
            loss = compiled_step(targets)
            mx.eval(model.parameters(), optimizer.state, loss)

        mx.clear_cache()

        elapsed = time.perf_counter() - start
        sps = num_steps / elapsed
        tps = sps * bs * seq_len
        latency_ms = (elapsed / num_steps) * 1000.0
        latencies[bs] = latency_ms / 1000.0

        print(f"{bs:<12} | {sps:<12.2f} | {tps:<15,.0f} | {latency_ms:<18.2f}")
        del model, optimizer, loss_and_grad_fn, loss
        gc.collect()
        mx.clear_cache()

    # 2. Precision Comparison Sweep (FP16 vs BF16 vs INT8 Quantized)
    eval_bs = 16 if model_size_key == "125M" else 32
    print("\n" + "=" * 85)
    print(f" 2. PRECISION BENCHMARK COMPARISON (Batch Size = {eval_bs}, Warmup={warmup_steps}, Steps={num_steps})")
    print("=" * 85)
    print(f"{'PRECISION':<12} | {'STEPS/SEC':<12} | {'TOKENS/SEC':<15} | {'LATENCY/STEP (ms)':<18} | {'RELATIVE SPEED'}")
    print("-" * 85)

    precisions = ["FP16", "BF16", "INT8 (8-bit)"]
    precision_tps = {}

    for prec in precisions:
        model = MLXTelosTransformer(**m_cfg)
        
        if prec == "FP16":
            model.set_dtype(mx.float16)
        elif prec == "BF16":
            model.set_dtype(mx.bfloat16)
        elif prec == "INT8 (8-bit)":
            model.set_dtype(mx.float16)
            nn.quantize(model, group_size=64, bits=8)
            
        mx.eval(model.parameters())

        optimizer = optim.AdamW(learning_rate=3e-4)
        loss_and_grad_fn = nn.value_and_grad(model, loss_fn)
        
        def compiled_step_uncompiled(tgts):
            masked_ids, mask_pos, t_vals = apply_masking_mlx(tgts)
            (loss, _), grads = loss_and_grad_fn(model, masked_ids, tgts, mask_pos, t_vals)
            optimizer.update(model, grads)
            return loss

        targets = mx.random.randint(0, 8192, (eval_bs, seq_len))
        mx.eval(targets)

        # Warmup and initialize state
        for _ in range(warmup_steps):
            loss = compiled_step_uncompiled(targets)
            mx.eval(model.parameters(), optimizer.state, loss)

        state = [model.state, optimizer.state]
        compiled_step = mx.compile(compiled_step_uncompiled, inputs=state, outputs=state)
        mx.metal.clear_cache()

        # Timed benchmark steps
        start = time.perf_counter()
        for _ in range(num_steps):
            loss = compiled_step(targets)
            mx.eval(model.parameters(), optimizer.state, loss)

        mx.metal.clear_cache()

        elapsed = time.perf_counter() - start
        sps = num_steps / elapsed
        tps = sps * eval_bs * seq_len
        latency_ms = (elapsed / num_steps) * 1000.0
        precision_tps[prec] = tps

        bf16_baseline = precision_tps.get("BF16", tps)
        rel_speed = f"{(tps / max(1.0, bf16_baseline)):.2f}x"

        print(f"{prec:<12} | {sps:<12.2f} | {tps:<15,.0f} | {latency_ms:<18.2f} | {rel_speed}")
        del model, optimizer, loss_and_grad_fn, loss
        gc.collect()
        mx.clear_cache()

    # 3. Effective Batch Size Sweep (1024 sequences)
    print("\n" + "=" * 85)
    print(f" 3. EFFECTIVE BATCH SIZE SWEEP (Fixed Eff Batch = 1024 seqs / 524,288 tok/step)")
    print("=" * 85)
    print(f"{'MICRO BATCH':<12} | {'GRAD ACCUM':<12} | {'EFF STEP (sec)':<15} | {'TOKENS/SEC':<15} | {'EFF STEPS/MIN':<15}")
    print("-" * 85)

    target_eff_batch = 1024
    eff_configs = [(8, 128), (16, 64), (32, 32)] if model_size_key == "125M" else [(16, 64), (32, 32), (64, 16)]

    for bs, grad_accum in eff_configs:
        micro_latency = latencies.get(bs, 0.15)
        sec_per_eff = micro_latency * grad_accum
        tps = (target_eff_batch * seq_len) / sec_per_eff
        eff_spm = 60.0 / sec_per_eff

        print(f"{bs:<12} | {grad_accum:<12} | {sec_per_eff:<15.2f} | {tps:<15,.0f} | {eff_spm:<15.2f}")

    # 4. Ratio Completion Times across Precisions
    print("\n" + "=" * 85)
    print(f" 4. PARAMETER:TOKEN RATIO COMPLETION TIMES ACROSS PRECISIONS ({model_size_key} Model)")
    print("=" * 85)
    print(f"{'PRECISION':<15} | {'1:0.5 Ratio':<18} | {'1:1.0 Ratio':<18} | {'1:2.0 Ratio':<18} | {'1:4.0 Ratio':<18}")
    print("-" * 85)

    ratios = [0.5, 1.0, 2.0, 4.0]

    for prec, tps in precision_tps.items():
        time_cols = []
        for mult in ratios:
            total_tok = int(param_count * mult)
            est_sec = total_tok / tps
            est_min = est_sec / 60.0
            if est_min < 60.0:
                time_cols.append(f"{est_min:.2f}m ({est_sec:.1f}s)")
            else:
                est_hrs = est_min / 60.0
                time_cols.append(f"{est_hrs:.2f}h ({est_min:.1f}m)")
        
        print(f"{prec:<15} | {time_cols[0]:<18} | {time_cols[1]:<18} | {time_cols[2]:<18} | {time_cols[3]:<18}")

    total_bench_time = time.perf_counter() - start_bench
    print("=" * 85)
    print(f"Benchmark Suite Completed in {total_bench_time:.2f} seconds!\n")


def tree_flatten(params):
    if isinstance(params, dict):
        for v in params.values():
            yield from tree_flatten(v)
    elif isinstance(params, list):
        for v in params:
            yield from tree_flatten(v)
    elif hasattr(params, "size"):
        yield params


if __name__ == "__main__":
    main()



 TELOS MDLM — 25M MODEL FAST BENCHMARK SUITE (Apple MLX Metal)
 Warmup Steps: 3 | Benchmark Steps: 15 | Metal Cache GC: Active
 Architecture: d_model=512, layers=8, heads=8, seq_len=512
 25M Model Parameter Count: 34,087,424

 1. MICROBATCH SIZE SWEEP (BF16 Baseline, Warmup=3, Steps=15)
BATCH SIZE   | STEPS/SEC    | TOKENS/SEC      | LATENCY/STEP (ms) 
-------------------------------------------------------------------------------------


16           | 6.39         | 52,313          | 156.59            


32           | 3.41         | 55,923          | 292.97            


64           | 1.76         | 57,830          | 566.62            

 2. PRECISION BENCHMARK COMPARISON (Batch Size = 32, Warmup=3, Steps=15)
PRECISION    | STEPS/SEC    | TOKENS/SEC      | LATENCY/STEP (ms)  | RELATIVE SPEED
-------------------------------------------------------------------------------------


mx.metal.clear_cache is deprecated and will be removed in a future version. Use mx.clear_cache instead.


FP16         | 3.41         | 55,844          | 293.39             | 1.00x


BF16         | 3.44         | 56,392          | 290.54             | 1.00x


INT8 (8-bit) | 2.78         | 45,535          | 359.81             | 0.81x

 3. EFFECTIVE BATCH SIZE SWEEP (Fixed Eff Batch = 1024 seqs / 524,288 tok/step)
MICRO BATCH  | GRAD ACCUM   | EFF STEP (sec)  | TOKENS/SEC      | EFF STEPS/MIN  
-------------------------------------------------------------------------------------
16           | 64           | 10.02           | 52,313          | 5.99           
32           | 32           | 9.38            | 55,923          | 6.40           
64           | 16           | 9.07            | 57,830          | 6.62           

 4. PARAMETER:TOKEN RATIO COMPLETION TIMES ACROSS PRECISIONS (25M Model)
PRECISION       | 1:0.5 Ratio        | 1:1.0 Ratio        | 1:2.0 Ratio        | 1:4.0 Ratio       
-------------------------------------------------------------------------------------
FP16            | 5.09m (305.2s)     | 10.17m (610.4s)    | 20.35m (1220.8s)   | 40.69m (2441.6s)  
BF16            | 5.04m (302.2s)     | 10.07m (604.5s)    | 20.15m (12

## Standard Benchmark
General profiling utility.

In [2]:
"""Unified Master Benchmarking Suite for télos MDLM.

Modes:
  --mode throughput : Measures generation & training throughput (tokens/sec).
  --mode samplers   : Compares Cosine vs Non-Monotonic vs Windowed samplers on prompt suite.
  --mode schedules  : Evaluates timestep schedule trade-offs across steps (16, 32, 64, 128).
"""

import sys
from pathlib import Path
sys.path.insert(0, "..")

import argparse
import time
import torch

from telos.hub.inference import TelosModel
from telos.diffusion.sampler import MDLMSampler, NonMonotonicMDLMSampler, WindowedMDLMSampler


PROMPT_SUITE = [
    "def fibonacci(n: int) -> int:\n    \"\"\"Return the nth Fibonacci number.\"\"\"\n",
    "def bubble_sort(arr: list) -> list:\n    \"\"\"Sort an array in ascending order.\"\"\"\n",
    "def read_json_file(file_path: str) -> dict:\n    \"\"\"Read and parse a JSON file.\"\"\"\n",
    "class Node:\n    def __init__(self, val=0, next=None):\n",
    "import math\n\ndef calculate_std_dev(data: list) -> float:\n",
]


def run_throughput_benchmark(model_obj: TelosModel):
    """Measures raw sampling throughput (tok/sec) on target device."""
    print("\n" + "=" * 80)
    print("RUNNING THROUGHPUT BENCHMARK")
    print("=" * 80)

    prompt = PROMPT_SUITE[0]
    steps_list = [16, 32, 64, 128]
    target_len = 64

    for steps in steps_list:
        # Warmup
        model_obj.complete(prompt, max_tokens=16, num_steps=16)

        t0 = time.time()
        res = model_obj.complete(prompt, max_tokens=target_len, num_steps=steps)
        elapsed = time.time() - t0

        tok_per_sec = target_len / elapsed
        print(f"Steps: {steps:3d} | Time: {elapsed:.2f}s | Throughput: {tok_per_sec:.1f} tok/s")


def run_sampler_comparison(model_obj: TelosModel):
    """Compares Cosine, Non-Monotonic, and Windowed samplers on prompt suite."""
    print("\n" + "=" * 80)
    print("RUNNING SAMPLER COMPARISON (Cosine vs Non-Monotonic vs Windowed)")
    print("=" * 80)

    model = model_obj.model
    tokenizer = model_obj.tokenizer
    mask_token_id = tokenizer.token_to_id("[MASK]") or 4

    cosine_sampler = MDLMSampler(model, mask_token_id, num_steps=64, schedule="cosine")
    non_mono_sampler = NonMonotonicMDLMSampler(model, mask_token_id, num_steps=64, remask_threshold=0.15)
    windowed_sampler = WindowedMDLMSampler(model, mask_token_id, window_size=32, num_steps_per_window=16)

    for i, prompt in enumerate(PROMPT_SUITE, 1):
        print(f"\n--- Prompt {i}: {prompt.splitlines()[0]} ---")
        prompt_enc = tokenizer.encode(prompt)
        prompt_ids = torch.tensor([prompt_enc.ids], device=model_obj.device)

        # 1. Cosine Sampler
        t0 = time.time()
        seq_cos = cosine_sampler.sample(seq_len=64 + len(prompt_enc.ids), prompt_ids=prompt_ids, device=model_obj.device)
        t_cos = time.time() - t0
        text_cos = tokenizer.decode(seq_cos[0].tolist())

        # 2. Non-Monotonic Sampler
        t0 = time.time()
        seq_nm = non_mono_sampler.sample(seq_len=64 + len(prompt_enc.ids), prompt_ids=prompt_ids, device=model_obj.device)
        t_nm = time.time() - t0
        text_nm = tokenizer.decode(seq_nm[0].tolist())

        # 3. Windowed Sampler
        t0 = time.time()
        seq_win = windowed_sampler.sample(target_tokens=64, prompt_ids=prompt_ids, device=model_obj.device)
        t_win = time.time() - t0
        text_win = tokenizer.decode(seq_win[0].tolist())

        print(f"[Cosine Sampler - {t_cos:.2f}s]:")
        print(text_cos[:150].replace("\n", " "))
        print(f"[Non-Monotonic - {t_nm:.2f}s]:")
        print(text_nm[:150].replace("\n", " "))
        print(f"[Windowed Sampler - {t_win:.2f}s]:")
        print(text_win[:150].replace("\n", " "))


def main():
    parser = argparse.ArgumentParser(description="Master Benchmarking CLI for télos MDLM")
    parser.add_argument("--checkpoint", type=str, default="checkpoints/phase_b_50m_1to10_mlx", help="Path to checkpoint")
    parser.add_argument("--mode", type=str, choices=["throughput", "samplers", "all"], default="all", help="Benchmark mode")
    args = parser.parse_known_args()[0]

    print(f"Loading checkpoint: {args.checkpoint}...")
    model_obj = TelosModel.from_pretrained(args.checkpoint)

    if args.mode in ["throughput", "all"]:
        run_throughput_benchmark(model_obj)

    if args.mode in ["samplers", "all"]:
        run_sampler_comparison(model_obj)


if __name__ == "__main__":
    main()


Loading checkpoint: checkpoints/phase_b_50m_1to10_mlx...


AssertionError: No model weights (.pt / .safetensors) found in checkpoints/phase_b_50m_1to10_mlx

## Deep Breakdown
Layer-by-layer MLX execution breakdown.

In [3]:
"""Deep Component & Sequence Length Benchmark Suite for MDLM (25M Model).

Benchmarks:
1. Phase Breakdown Timing:
   - Forward pass only
   - Forward + Backward pass
   - Forward + Backward + AdamW Optimizer step
2. Sequence Length Breakdown Timing (128, 256, 512, 1024)
"""

import sys
import time
import argparse
from pathlib import Path
import numpy as np
import mlx.core as mx
import mlx.nn as nn
import mlx.optimizers as optim

sys.path.insert(0, "..")


class MLXRMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-5):
        super().__init__()
        self.weight = mx.ones((d_model,))
        self.eps = eps

    def __call__(self, x):
        variance = mx.mean(mx.square(x), axis=-1, keepdims=True)
        return x * mx.rsqrt(variance + self.eps) * self.weight


class MLXSwiGLU(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        hidden = int(d_model * 8 / 3)
        hidden = ((hidden + 63) // 64) * 64
        self.w1 = nn.Linear(d_model, hidden, bias=False)
        self.w2 = nn.Linear(d_model, hidden, bias=False)
        self.w3 = nn.Linear(hidden, d_model, bias=False)

    def __call__(self, x):
        return self.w3(nn.silu(self.w1(x)) * self.w2(x))


class MLXBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        self.norm1 = MLXRMSNorm(d_model)
        self.norm2 = MLXRMSNorm(d_model)
        self.head_dim = d_model // n_heads
        self.n_heads = n_heads

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out = nn.Linear(d_model, d_model, bias=False)
        self.mlp = MLXSwiGLU(d_model)

    def __call__(self, x):
        B, T, D = x.shape
        h = self.norm1(x)

        q = self.q_proj(h).reshape(B, T, self.n_heads, self.head_dim).transpose(0, 2, 1, 3)
        k = self.k_proj(h).reshape(B, T, self.n_heads, self.head_dim).transpose(0, 2, 1, 3)
        v = self.v_proj(h).reshape(B, T, self.n_heads, self.head_dim).transpose(0, 2, 1, 3)

        scale = 1.0 / (self.head_dim ** 0.5)
        out = mx.fast.scaled_dot_product_attention(q, k, v, scale=scale)
        out = out.transpose(0, 2, 1, 3).reshape(B, T, D)

        x = x + self.out(out)
        x = x + self.mlp(self.norm2(x))
        return x


class MLXTelosTransformer25M(nn.Module):
    def __init__(self, vocab_size=8192, d_model=512, n_layers=8, n_heads=8):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model)
        self.layers = [MLXBlock(d_model, n_heads) for _ in range(n_layers)]
        self.norm = MLXRMSNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def __call__(self, x):
        x = self.emb(x)
        for layer in self.layers:
            x = layer(x)
        return self.head(self.norm(x))


def sample_beta_timesteps(batch_size: int, eps: float = 1e-5):
    u = mx.random.uniform(0.0, 1.0, (batch_size, 1))
    t = 0.5 - 0.5 * mx.cos(np.pi * u)
    return mx.clip(t, eps, 1.0)


def apply_masking_mlx(input_ids, mask_token_id=1):
    B, T = input_ids.shape
    t_values = sample_beta_timesteps(B)
    rand_matrix = mx.random.uniform(0.0, 1.0, (B, T))
    mask_positions = rand_matrix < t_values
    masked_input_ids = mx.where(mask_positions, mask_token_id, input_ids)
    return masked_input_ids, mask_positions, t_values


def loss_fn(model, masked_input_ids, targets, mask_positions, t_values):
    logits = model(masked_input_ids)
    B, T, V = logits.shape

    logits_flat = logits.reshape(-1, V)
    targets_flat = targets.reshape(-1)

    ce_per_token = nn.losses.cross_entropy(logits_flat, targets_flat, reduction="none").reshape(B, T)
    masked_ce = ce_per_token * mask_positions.astype(mx.float32)

    masked_count = mx.clip(mx.sum(mask_positions.astype(mx.float32), axis=1), 1.0, float(T))
    per_example_ce = mx.sum(masked_ce, axis=1) / masked_count

    t_weights = 1.0 / mx.clip(mx.squeeze(t_values, -1), 1e-3, 1.0)
    reweighted_loss = mx.mean(per_example_ce * t_weights)
    return reweighted_loss


def run_phase_breakdown_benchmark(model, targets, num_steps=20):
    print("\n" + "=" * 85)
    print(" 1. PHASE BREAKDOWN TIMING (Forward vs Forward+Backward vs Forward+Backward+AdamW)")
    print("=" * 85)
    print(f"{'PHASE STAGE':<35} | {'LATENCY (ms)':<15} | {'PERCENT OF TOTAL':<18} | {'TOKENS/SEC':<15}")
    print("-" * 85)

    bs, seq_len = targets.shape
    optimizer = optim.AdamW(learning_rate=3e-4)
    loss_and_grad_fn = nn.value_and_grad(model, loss_fn)

    # A) Forward Only
    for _ in range(3):
        masked_ids, mask_pos, t_vals = apply_masking_mlx(targets)
        loss = loss_fn(model, masked_ids, targets, mask_pos, t_vals)
        mx.eval(loss)

    mx.clear_cache()
    start = time.perf_counter()
    for _ in range(num_steps):
        masked_ids, mask_pos, t_vals = apply_masking_mlx(targets)
        loss = loss_fn(model, masked_ids, targets, mask_pos, t_vals)
        mx.eval(loss)
    fwd_latency = ((time.perf_counter() - start) / num_steps) * 1000.0

    # B) Forward + Backward
    for _ in range(3):
        masked_ids, mask_pos, t_vals = apply_masking_mlx(targets)
        loss, grads = loss_and_grad_fn(model, masked_ids, targets, mask_pos, t_vals)
        mx.eval(loss, grads)

    mx.clear_cache()
    start = time.perf_counter()
    for _ in range(num_steps):
        masked_ids, mask_pos, t_vals = apply_masking_mlx(targets)
        loss, grads = loss_and_grad_fn(model, masked_ids, targets, mask_pos, t_vals)
        mx.eval(loss, grads)
    fwd_bwd_latency = ((time.perf_counter() - start) / num_steps) * 1000.0

    # C) Forward + Backward + AdamW Optimizer Step
    for _ in range(3):
        masked_ids, mask_pos, t_vals = apply_masking_mlx(targets)
        loss, grads = loss_and_grad_fn(model, masked_ids, targets, mask_pos, t_vals)
        optimizer.update(model, grads)
        mx.eval(model.parameters(), optimizer.state)

    mx.clear_cache()
    start = time.perf_counter()
    for _ in range(num_steps):
        masked_ids, mask_pos, t_vals = apply_masking_mlx(targets)
        loss, grads = loss_and_grad_fn(model, masked_ids, targets, mask_pos, t_vals)
        optimizer.update(model, grads)
        mx.eval(model.parameters(), optimizer.state)
    total_latency = ((time.perf_counter() - start) / num_steps) * 1000.0

    bwd_alone = fwd_bwd_latency - fwd_latency
    adamw_alone = max(0.0, total_latency - fwd_bwd_latency)

    fwd_pct = (fwd_latency / total_latency) * 100.0
    bwd_pct = (bwd_alone / total_latency) * 100.0
    adamw_pct = (adamw_alone / total_latency) * 100.0

    fwd_tps = (bs * seq_len) / (fwd_latency / 1000.0)
    total_tps = (bs * seq_len) / (total_latency / 1000.0)

    print(f"{'Forward Pass Only':<35} | {fwd_latency:<15.2f} | {fwd_pct:<17.1f}% | {fwd_tps:<15,.0f}")
    print(f"{'Backward Pass Only (Grad Comp)':<35} | {bwd_alone:<15.2f} | {bwd_pct:<17.1f}% | {'N/A':<15}")
    print(f"{'AdamW Optimizer Step Only':<35} | {adamw_alone:<15.2f} | {adamw_pct:<17.1f}% | {'N/A':<15}")
    print(f"{'TOTAL FULL STEP (Fwd + Bwd + AdamW)':<35} | {total_latency:<15.2f} | {'100.0%':<18} | {total_tps:<15,.0f}")


def run_sequence_length_benchmark(model, num_steps=15):
    print("\n" + "=" * 85)
    print(" 2. SEQUENCE LENGTH BREAKDOWN TIMING (Batch Size = 16, BF16)")
    print("=" * 85)
    print(f"{'SEQ LENGTH':<12} | {'STEPS/SEC':<12} | {'TOKENS/SEC':<15} | {'LATENCY/STEP (ms)':<18} | {'RELATIVE FLOPs'}")
    print("-" * 85)

    seq_lengths = [128, 256, 512, 1024]
    bs = 16
    base_latency = None

    for seq_len in seq_lengths:
        targets = mx.random.randint(0, 8192, (bs, seq_len))
        mx.eval(targets)
        optimizer = optim.AdamW(learning_rate=3e-4)
        loss_and_grad_fn = nn.value_and_grad(model, loss_fn)

        # Warmup
        for _ in range(3):
            masked_ids, mask_pos, t_vals = apply_masking_mlx(targets)
            loss, grads = loss_and_grad_fn(model, masked_ids, targets, mask_pos, t_vals)
            optimizer.update(model, grads)
            mx.eval(model.parameters(), optimizer.state)

        mx.clear_cache()
        start = time.perf_counter()

        for _ in range(num_steps):
            masked_ids, mask_pos, t_vals = apply_masking_mlx(targets)
            loss, grads = loss_and_grad_fn(model, masked_ids, targets, mask_pos, t_vals)
            optimizer.update(model, grads)
            mx.eval(model.parameters(), optimizer.state)

        mx.clear_cache()
        elapsed = time.perf_counter() - start
        sps = num_steps / elapsed
        tps = sps * bs * seq_len
        latency_ms = (elapsed / num_steps) * 1000.0

        if base_latency is None:
            base_latency = latency_ms

        rel_ratio = f"{(latency_ms / base_latency):.2f}x"
        print(f"{seq_len:<12} | {sps:<12.2f} | {tps:<15,.0f} | {latency_ms:<18.2f} | {rel_ratio}")

    print("=" * 85 + "\n")


def main():
    print("\n" + "=" * 85)
    print(" TELOS MDLM — DEEP COMPONENT & SEQUENCE LENGTH BENCHMARK (25M Model)")
    print(" Device: Apple MLX Metal | Precision: bfloat16 | Clean int32 Input Feeding")
    print("=" * 85)

    model = MLXTelosTransformer25M()
    model.set_dtype(mx.bfloat16)
    mx.eval(model.parameters())

    targets = mx.random.randint(0, 8192, (32, 512))
    mx.eval(targets)

    start_total = time.perf_counter()
    run_phase_breakdown_benchmark(model, targets, num_steps=20)
    run_sequence_length_benchmark(model, num_steps=15)

    print(f"Deep Breakdown Benchmark Suite Completed in {time.perf_counter() - start_total:.2f} seconds!\n")


if __name__ == "__main__":
    main()



 TELOS MDLM — DEEP COMPONENT & SEQUENCE LENGTH BENCHMARK (25M Model)
 Device: Apple MLX Metal | Precision: bfloat16 | Clean int32 Input Feeding

 1. PHASE BREAKDOWN TIMING (Forward vs Forward+Backward vs Forward+Backward+AdamW)
PHASE STAGE                         | LATENCY (ms)    | PERCENT OF TOTAL   | TOKENS/SEC     
-------------------------------------------------------------------------------------


Forward Pass Only                   | 69.80           | 17.0             % | 234,743        
Backward Pass Only (Grad Comp)      | 249.85          | 60.8             % | N/A            
AdamW Optimizer Step Only           | 91.44           | 22.2             % | N/A            
TOTAL FULL STEP (Fwd + Bwd + AdamW) | 411.08          | 100.0%             | 39,856         

 2. SEQUENCE LENGTH BREAKDOWN TIMING (Batch Size = 16, BF16)
SEQ LENGTH   | STEPS/SEC    | TOKENS/SEC      | LATENCY/STEP (ms)  | RELATIVE FLOPs
-------------------------------------------------------------------------------------


128          | 24.44        | 50,048          | 40.92              | 1.00x


256          | 11.54        | 47,278          | 86.64              | 2.12x


512          | 4.51         | 36,956          | 221.67             | 5.42x


1024         | 1.74         | 28,458          | 575.72             | 14.07x

Deep Breakdown Benchmark Suite Completed in 35.83 seconds!



## Fine Breakdown
Detailed operation-level profiling.

In [4]:
"""Fine-grained MDLM Benchmark Suite: Forward, Loss, and Backward Attribution.

Isolates:
1. Model Forward Pass: input_ids -> logits
2. MDLM Loss Computation: logits -> loss scalar
3. Backward Pass: loss -> parameter gradients
4. AdamW Optimizer Update: gradients -> updated weights
"""

import sys
import time
from pathlib import Path
import numpy as np
import mlx.core as mx
import mlx.nn as nn
import mlx.optimizers as optim

sys.path.insert(0, "..")


class MLXRMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-5):
        super().__init__()
        # Gain parameter initialized to ones
        self.weight = mx.ones((d_model,))
        self.eps = eps

    def __call__(self, x):
        # RMS normalization along the hidden dimension
        variance = mx.mean(mx.square(x), axis=-1, keepdims=True)
        return x * mx.rsqrt(variance + self.eps) * self.weight


class MLXSwiGLU(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        # SwiGLU hidden dimension calculation matching LLaMA / Telos architecture
        hidden = int(d_model * 8 / 3)
        hidden = ((hidden + 63) // 64) * 64
        self.w1 = nn.Linear(d_model, hidden, bias=False)
        self.w2 = nn.Linear(d_model, hidden, bias=False)
        self.w3 = nn.Linear(hidden, d_model, bias=False)

    def __call__(self, x):
        # SwiGLU activation: (SiLU(w1(x)) * w2(x)) @ w3
        return self.w3(nn.silu(self.w1(x)) * self.w2(x))


class MLXBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        self.norm1 = MLXRMSNorm(d_model)
        self.norm2 = MLXRMSNorm(d_model)
        self.head_dim = d_model // n_heads
        self.n_heads = n_heads

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out = nn.Linear(d_model, d_model, bias=False)
        self.mlp = MLXSwiGLU(d_model)

    def __call__(self, x):
        B, T, D = x.shape
        h = self.norm1(x)

        # Multi-head attention projection and transpose
        q = self.q_proj(h).reshape(B, T, self.n_heads, self.head_dim).transpose(0, 2, 1, 3)
        k = self.k_proj(h).reshape(B, T, self.n_heads, self.head_dim).transpose(0, 2, 1, 3)
        v = self.v_proj(h).reshape(B, T, self.n_heads, self.head_dim).transpose(0, 2, 1, 3)

        scale = 1.0 / (self.head_dim ** 0.5)
        # Scaled dot-product attention
        out = mx.fast.scaled_dot_product_attention(q, k, v, scale=scale)
        out = out.transpose(0, 2, 1, 3).reshape(B, T, D)

        x = x + self.out(out)
        x = x + self.mlp(self.norm2(x))
        return x


class MLXTelosTransformer25M(nn.Module):
    def __init__(self, vocab_size=8192, d_model=512, n_layers=8, n_heads=8):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model)
        self.layers = [MLXBlock(d_model, n_heads) for _ in range(n_layers)]
        self.norm = MLXRMSNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def __call__(self, x):
        x = self.emb(x)
        for layer in self.layers:
            x = layer(x)
        return self.head(self.norm(x))


def compute_mdlm_loss_only(logits, targets, mask_positions, t_values):
    """Calculates the discrete diffusion cross-entropy loss from logits."""
    B, T, V = logits.shape
    logits_flat = logits.reshape(-1, V)
    targets_flat = targets.reshape(-1)

    # Compute unweighted per-token cross entropy
    ce_per_token = nn.losses.cross_entropy(logits_flat, targets_flat, reduction="none").reshape(B, T)
    masked_ce = ce_per_token * mask_positions.astype(mx.float32)

    # Normalize by masked tokens count per sequence
    masked_count = mx.clip(mx.sum(mask_positions.astype(mx.float32), axis=1), 1.0, float(T))
    per_example_ce = mx.sum(masked_ce, axis=1) / masked_count

    # 1/t timestep reweighting
    t_weights = 1.0 / mx.clip(mx.squeeze(t_values, -1), 1e-3, 1.0)
    return mx.mean(per_example_ce * t_weights)


def combined_forward_and_loss(model, masked_input_ids, targets, mask_positions, t_values):
    """Combined forward and loss function for value_and_grad calculation."""
    logits = model(masked_input_ids)
    return compute_mdlm_loss_only(logits, targets, mask_positions, t_values)


def run_fine_grained_attribution(model, targets, num_steps=25):
    print("\n" + "=" * 90)
    print(" FINE-GRAINED COMPONENT ATTRIBUTION: Model Forward vs Loss Computation vs Backward")
    print("=" * 90)

    bs, seq_len = targets.shape
    mask_token_id = 1
    optimizer = optim.AdamW(learning_rate=3e-4)

    # Create dynamic input sample
    u = mx.random.uniform(0.0, 1.0, (bs, 1))
    t_values = mx.clip(0.5 - 0.5 * mx.cos(np.pi * u), 1e-5, 1.0)
    rand_matrix = mx.random.uniform(0.0, 1.0, (bs, seq_len))
    mask_positions = rand_matrix < t_values
    masked_input_ids = mx.where(mask_positions, mask_token_id, targets)
    mx.eval(masked_input_ids, mask_positions, t_values)

    # Warmup calls
    for _ in range(5):
        logits = model(masked_input_ids)
        loss = compute_mdlm_loss_only(logits, targets, mask_positions, t_values)
        mx.eval(logits, loss)

    # 1. Benchmark Model Forward Pass Only (input_ids -> logits)
    mx.clear_cache()
    start = time.perf_counter()
    for _ in range(num_steps):
        logits = model(masked_input_ids)
        mx.eval(logits)
    fwd_model_latency = ((time.perf_counter() - start) / num_steps) * 1000.0

    # 2. Benchmark Loss Computation Only (logits -> loss)
    mx.clear_cache()
    start = time.perf_counter()
    for _ in range(num_steps):
        loss = compute_mdlm_loss_only(logits, targets, mask_positions, t_values)
        mx.eval(loss)
    loss_calc_latency = ((time.perf_counter() - start) / num_steps) * 1000.0

    # Total Forward Time (Model Forward + Loss Computation)
    total_fwd_latency = fwd_model_latency + loss_calc_latency

    # 3. Benchmark Forward + Backward (Total Forward + Autograd Backward)
    grad_fn = nn.value_and_grad(model, combined_forward_and_loss)
    # Warmup
    for _ in range(5):
        loss, grads = grad_fn(model, masked_input_ids, targets, mask_positions, t_values)
        mx.eval(loss, grads)

    mx.clear_cache()
    start = time.perf_counter()
    for _ in range(num_steps):
        loss, grads = grad_fn(model, masked_input_ids, targets, mask_positions, t_values)
        mx.eval(loss, grads)
    fwd_bwd_latency = ((time.perf_counter() - start) / num_steps) * 1000.0

    # Isolated Backward Pass (Grad Computation across all 59 Transformer weight matrices)
    bwd_only_latency = fwd_bwd_latency - total_fwd_latency

    # 4. Benchmark Full Step (Forward + Backward + AdamW Optimizer Update)
    for _ in range(5):
        loss, grads = grad_fn(model, masked_input_ids, targets, mask_positions, t_values)
        optimizer.update(model, grads)
        mx.eval(model.parameters(), optimizer.state)

    mx.clear_cache()
    start = time.perf_counter()
    for _ in range(num_steps):
        loss, grads = grad_fn(model, masked_input_ids, targets, mask_positions, t_values)
        optimizer.update(model, grads)
        mx.eval(model.parameters(), optimizer.state)
    full_step_latency = ((time.perf_counter() - start) / num_steps) * 1000.0

    adamw_latency = max(0.0, full_step_latency - fwd_bwd_latency)

    # Compute percentages relative to full step
    fwd_model_pct = (fwd_model_latency / full_step_latency) * 100.0
    loss_calc_pct = (loss_calc_latency / full_step_latency) * 100.0
    bwd_only_pct = (bwd_only_latency / full_step_latency) * 100.0
    adamw_pct = (adamw_latency / full_step_latency) * 100.0

    print(f"{'COMPONENT / PHASE':<42} | {'LATENCY (ms)':<15} | {'PERCENTAGE':<15}")
    print("-" * 90)
    print(f"{'1. Model Forward Pass (Layers + Projections)':<42} | {fwd_model_latency:<15.2f} | {fwd_model_pct:<14.1f}%")
    print(f"{'2. Loss Computation Only (CE + Reweighting)':<42} | {loss_calc_latency:<15.2f} | {loss_calc_pct:<14.1f}%")
    print(f"{'   -> TOTAL FORWARD (Model + Loss)':<42} | {total_fwd_latency:<15.2f} | {(total_fwd_latency/full_step_latency)*100:<14.1f}%")
    print(f"{'3. Autograd Backward Pass (Gradient Compute)':<42} | {bwd_only_latency:<15.2f} | {bwd_only_pct:<14.1f}%")
    print(f"{'4. AdamW Optimizer Step (Weight Updates)':<42} | {adamw_latency:<15.2f} | {adamw_pct:<14.1f}%")
    print("-" * 90)
    print(f"{'TOTAL TRAIN STEP TIME':<42} | {full_step_latency:<15.2f} | {'100.0%':<15}")
    print("=" * 90 + "\n")


def main():
    print("\n" + "=" * 90)
    print(" TELOS 25M MDLM — FINE-GRAINED BENCHMARK ATTRIBUTION")
    print(" Precision: bfloat16 | Batch Size: 32 | Seq Length: 512 | Total Tokens/Step: 16,384")
    print("=" * 90)

    model = MLXTelosTransformer25M()
    model.set_dtype(mx.bfloat16)
    mx.eval(model.parameters())

    targets = mx.random.randint(0, 8192, (32, 512))
    mx.eval(targets)

    run_fine_grained_attribution(model, targets, num_steps=25)


if __name__ == "__main__":
    main()



 TELOS 25M MDLM — FINE-GRAINED BENCHMARK ATTRIBUTION
 Precision: bfloat16 | Batch Size: 32 | Seq Length: 512 | Total Tokens/Step: 16,384

 FINE-GRAINED COMPONENT ATTRIBUTION: Model Forward vs Loss Computation vs Backward


COMPONENT / PHASE                          | LATENCY (ms)    | PERCENTAGE     
------------------------------------------------------------------------------------------
1. Model Forward Pass (Layers + Projections) | 114.30          | 20.4          %
2. Loss Computation Only (CE + Reweighting) | 1.76            | 0.3           %
   -> TOTAL FORWARD (Model + Loss)         | 116.06          | 20.7          %
3. Autograd Backward Pass (Gradient Compute) | 351.52          | 62.7          %
4. AdamW Optimizer Step (Weight Updates)   | 92.70           | 16.5          %
------------------------------------------------------------------------------------------
TOTAL TRAIN STEP TIME                      | 560.28          | 100.0%         

